In [0]:

source_path = "/Volumes/banking_project/bronze/banking_raw/transactions/"

schema_path = "/Volumes/banking_project/bronze/autoloader_metadata/schemas/transactions/"

checkpoint_path = "/Volumes/banking_project/bronze/autoloader_metadata/checkpoints/transactions/"

target_table = "banking_project.bronze.bronze_transactions"


In [0]:

from pyspark.sql.functions import current_timestamp, col

df_transactions = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("cloudFiles.useManagedFileEvents", "false")
    .load(source_path)
)

df_bronze = (
    df_transactions
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)


In [0]:

query = (
    df_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()


In [0]:
spark.sql("""
    SELECT COUNT(*) AS total_records
    FROM banking_project.bronze.bronze_transactions
""").show()